# 14. 색상별 일반화 성능비교

실제 SegFormer 추론 결과를 color group별로 비교합니다.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch2_utils.py").exists():
    matches = (
        list(Path.cwd().glob("Deeplearning/*/2-1장/ch2_utils.py"))
        + list(Path.cwd().glob("Deeplearning/*/2장/ch2_utils.py"))
        + list(Path.cwd().glob("**/ch2_utils.py"))
    )
    NOTEBOOK_DIR = matches[0].parent if matches else Path("Deeplearning") / "Vision 응용" / "2-1장"
sys.path.append(str(NOTEBOOK_DIR))

from ch2_utils import *

paths = find_paths()
set_korean_font()
set_seed(7)
samples = load_samples(paths.data_root)
paths

In [ ]:
run_dir = paths.runs_root / "baseline_segformer_b0"
sample_metrics, group_metrics, class_metrics = load_run_metrics(run_dir)

## 14-1. 색상별 target Dice

In [ ]:
color_table = plot_metric_bar(
    group_metrics,
    grouping="color_group",
    metric="target_dice_mean",
    title="색상별 defect Dice",
    out_path=paths.runs_root / "14_color_target_dice.png",
)
display(color_table)

## 14-2. Color x Defect heatmap

In [ ]:
pivot = plot_metric_heatmap(
    sample_metrics,
    row="color_group",
    col="defect_type",
    metric="target_dice",
    title="color x defect target Dice",
    out_path=paths.runs_root / "14_color_defect_heatmap.png",
)
display(pivot)

## 14-3. 색상 효과 결론

In [ ]:
print(conclusion_from_group(group_metrics, "color_group", "target_dice_mean"))
available = set(sample_metrics["color_group"])
if {"red", "blue"}.issubset(available):
    result = compare_two_groups(sample_metrics, "color_group", "red", "blue", "target_dice")
    display(pd.DataFrame([result]))
    print(
        "결론:",
        "red와 blue의 95% bootstrap CI가 0을 포함하지 않습니다." if result["reject_h0_ci_excludes_0"]
        else "red와 blue의 차이는 현재 반복/샘플 기준에서 95% CI로 확정되지 않았습니다."
    )